<a href="https://colab.research.google.com/github/elope237/CAHSI-26research/blob/main/Jupyter_Notebooks/QAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantum Amplitude Estimation
There are many different variations of the original QAE algorithm but they all have the same goal, to use random sampling to estimate quantum amplitudes and probabilites. The classical version is Monte Carlo Simulation.
It has applications in finance option pricing, integration, and anything else that uses random sampling.

The problem goes like this, lets assume we are given an operator A, we then apply it to a quantum system and want to measure how it changes the ampltidues.

We start by applying A to our qubit to change it into the $\lvert{\psi}\rangle$ state.
$$A\lvert{0}\rangle = \lvert{\psi}\rangle$$
We then mark the "success" or the state we want to measure the amplitude of.
$$\lvert{\psi}\rangle = \sqrt{1-p}\lvert{\psi_\text{bad}}\rangle + \sqrt{p}\lvert{\psi_{\text{good}}}\rangle$$

or more formally

$$A\lvert{\Psi_0}\rangle\lvert0\rangle = \sqrt{1-a}\lvert{\Psi_0}\rangle\lvert0\rangle + \sqrt{a}\lvert{\Psi_1}\rangle\lvert1\rangle$$


$p_k = \sin^2\left((2k+1)\theta\right), \quad \theta = \arcsin\left(\sqrt{a}\right)$

The classical way to compute this using monte carlo simulation. This has an error rate with scaling of $\frac{1}{\sqrt{M}}$. Thankfully there is also a way to compute this quantumaly. When we do this, we are able to compute this with a error rate scaling of $\frac{1}{M}$. For this explanation we will use a version of QAE called Iterative Quantum Amplitdue Estimation (IQAE).

## QAE Applications
QAE algorihtms have been shown to be able to achieve expontential speed up over classical computing Monte Carlo simulations. This is useful for the areas of high dimensional integration, finance model derivations, and quantum physics.


## QAE Example Runthrough

As mentioned above, one application of QAE algorithms is that they provide a speedup in the approximation of complete integrals. Lets go through how this is the case.
Given an integral:
$$ I = \int f(x) dx $$
we can rewrite it as:
$$I = \int p(x)\, g(x)\, dx = \mathbb{E}_p[g(x)]$$
where $p(x)$ is a probability distribution and $g(x)$ is $\frac{f(x)}{p(x)}$. $\mathbb{E}_p[g(x)]$ is the expectation operator of $g(x)$, it can be though of as the average value of $g(x)$ given we are sampling over a distribution $p(x)$. The integral over $f(x)$ can be estimated by collecting the expecation values of integrating over different probility distributions.

Given our formula for QAE defined above:
$$A\lvert{\Psi_0}\rangle\lvert0\rangle = \sqrt{1-a}\lvert{\Psi_0}\rangle\lvert0\rangle + \sqrt{a}\lvert{\Psi_1}\rangle\lvert1\rangle$$
we can think of $\lvert{\Psi_1}\rangle$ as being the "good" state we are trying to measure and in the context of integration approximation this means

$$
\lvert\Psi_1\rangle =
\frac{1}{\sqrt{a}} \sum_x \sqrt{p(x)g(x)} \, |x\rangle_n
$$

The original QAE then procedes by defining an Oracle then applying it $M$ times where $M = 2^m$ where $m$ is the number of ancilla qubits. It then applied the inverse Quantum Fourier Transform to the ancilla qubits and the result obtained from the measurement is $y \in \{0, \ldots, M-1\}$ and the estimation of $a$ is $\tilde{a} = \sin^2(\theta_a)$ where $\theta_a \equiv\frac{y\pi}{M}$ with error scaling at $ \sim \frac{1}{M}$.


One drawback of the original QAE algorithm is that it requires the use of the Quantum Fourier Transform (QFT) and multiple ancilla qubits. Iterative Quantum Amplitude Estimation (IQAE) removes them and instead uses a classical iterative subroutine that repeatedly refines an estimate of the amplitude.

As in standard QAE, we begin with the state
$$A|\Psi_0\rangle|0\rangle = \sqrt{1-a}|\Psi_0\rangle|0\rangle
+ \sqrt{a}|\Psi_1\rangle|1\rangle$$

where the goal is to estimate the amplitude $a$, corresponding to the probability of measuring the “good” state $|1\rangle$.

Instead of performing phase estimation, IQAE repeatedly applies powers of the Grover operator $Q$ and measures the resulting probability of observing the good state. After applying $Q^k$, the probability of measuring $|1\rangle$ in the final qubit is

$$P(|1\rangle) = \sin^2((2k+1)\theta_a) $$
where the amplitude $a$ is related to the angle $\theta_a$ by $a = \sin^2(\theta_a)$

To estimate this probability, the circuit

$$Q^k A |0\rangle_n |0\rangle$$

is executed multiple times and the frequency of measuring $|1\rangle$ is recorded.

The algorithm uses a confidence interval $[\theta_l, \theta_u]$ that contains the true value of $\theta_a$.

At each iteration, a classical routine chooses an integer $k$ that maximizes the information obtained from the measurement while ensuring that the transformed interval $[(4k+2)\theta_l,(4k+2)\theta_u] \bmod 2\pi$ lies entirely within either $[0,\pi] \text{ or } [\pi,2\pi]$.

This constraint ensures that the function $\cos((4k+2)\theta_a)$ can be uniquely inverted, allowing the algorithm to shrink the confidence interval for $\theta_a$.

By repeating this process iteratively, the interval containing $\theta_a$ becomes progressively smaller until the desired accuracy $\epsilon$ is reached. The final estimate of the amplitude is then obtained from $\tilde{a} = \sin^2(\theta_a)$.

Unlike the original QAE algorithm, IQAE does not require the QFT or large ancilla registers. Instead, it relies on repeated circuit executions and classical post-processing to refine the estimate while still achieving the quadratic speedup over classical Monte Carlo methods.

In [ ]:
# @title Initial Imports
!pip install -q qiskit
!pip install -q qiskit_algorithms
!pip install -q qiskit_aer

Below the intervals are shown for different amounts of Grover Oracle calls, the green lines plot a $\frac{1}{M}$ relationship and we can see that the bounds of the confidence intervals are approaching the green lines.

In [ ]:
# @title QAE Error Plot
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_algorithms import IterativeAmplitudeEstimation, EstimationProblem
from qiskit_aer.primitives import SamplerV2

# -----------------------------
# Bernoulli operator setup
# -----------------------------
p_true = 0.35

class BernoulliA(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(theta_p, 0)

class BernoulliQ(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        self._theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(2 * self._theta_p, 0)
    def power(self, k):
        qc = QuantumCircuit(1)
        qc.ry(2 * k * self._theta_p, 0)
        qc.metadata = {"k": k}  # store k for oracle count tracking
        return qc


# -----------------------------
# Prepare IQAE problem
# -----------------------------
A = BernoulliA(p_true)
Q = BernoulliQ(p_true)

problem = EstimationProblem(
    state_preparation=A,
    grover_operator=Q,
    objective_qubits=[0]
)

sampler = SamplerV2()

# -----------------------------
# Run IQAE for multiple epsilons
# -----------------------------
# Desired range of oracle calls
min_calls = 500
max_calls = 1000
num_points = 50

# Generate oracle calls evenly in linear space
target_oracle_calls = np.linspace(min_calls, max_calls, num_points)

# Convert back to epsilon values (roughly ε ~ 1 / M)
epsilon_list = 1 / target_oracle_calls
#epsilon_list = np.logspace(-1, -3.3, 400)
estimates = []
ci_lowers = []
ci_uppers = []
oracle_calls = []

for eps in epsilon_list:

    iae = IterativeAmplitudeEstimation(
        epsilon_target=eps,
        alpha=0.05,
        sampler=sampler
    )
    result = iae.estimate(problem)
    if (result.num_oracle_queries in oracle_calls):
        continue  # skip if this oracle call count has already been recorded

    estimates.append(result.estimation)
    ci_lowers.append(result.confidence_interval[0])
    ci_uppers.append(result.confidence_interval[1])
    oracle_calls.append(result.num_oracle_queries)  # exact oracle calls

# Prepare data to be plotted

estimates = np.array(estimates)
ci_lowers = np.array(ci_lowers)
ci_uppers = np.array(ci_uppers)
errors = np.stack([estimates - ci_lowers, ci_uppers - estimates])
oracle_calls = np.array(oracle_calls)

# Compute absolute errors from true probability
errors = np.stack([estimates - ci_lowers, ci_uppers - estimates])

# Remove entries with 0 oracle calls
nonzero_mask = oracle_calls > 0
oracle_calls_nonzero = oracle_calls[nonzero_mask]
errors_nonzero = errors[:, nonzero_mask]
estimates_nonzero = estimates[nonzero_mask]


# Reference 1/M line

# Scale the line to match the first nonzero error
scale_factor = errors_nonzero[1,0]  # upper CI width
iqae_ref = scale_factor * oracle_calls_nonzero[0] / oracle_calls_nonzero


# Plot

plt.figure(figsize=(10,4))

# IQAE estimates with CI
plt.errorbar(
    x=oracle_calls_nonzero,
    y=estimates_nonzero,
    yerr=errors_nonzero,
    fmt='o',
    capsize=4,
    label='IQAE estimate ± CI'
)

# True probability
plt.axhline(p_true, color='r', linestyle='--', label='True probability')

# Ideal IQAE 1/M reference
plt.plot(oracle_calls_nonzero, p_true + iqae_ref, 'g-.', label='1/M scaling')
plt.plot(oracle_calls_nonzero, p_true - iqae_ref, 'g-.')

plt.xscale('log')
plt.yscale('linear')
plt.xlabel('Number of oracle calls $M$', fontsize=12)
plt.ylabel('Estimated amplitude', fontsize=12)
plt.title('IQAE Estimates with Exact Oracle Calls (1/M Scaling)', fontsize=14)
plt.grid(True, which='both', ls='--', alpha=0.5)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

IQAE is $ \text{error} \sim \frac{1}{M}$ so $M \sim \frac{1}{\varepsilon}\quad\text{(oracle calls)} $
This shows that the confidence intervals are becoming bounded by the error